# 스프린트 미션 15 · 연구자 1 — 데이터 전처리 · EDA · 모델링

**목표변수:** `Performance Index` (학생 성취도 지수, 10~100)  
**과제:** `train.csv` 로 회귀 모델을 학습하고 RMSE 로 평가한 뒤 `model.pkl` 로 저장한다.

이 노트북에서 확정한 전처리·모델링 파이프라인을 그대로 `train.py` 로 옮겨 도커 이미지로 자동화한다.

## 1. 라이브러리 및 데이터 로드

In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Colab에서 Google Drive 마운트 (옵션)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
    print("로컬 환경에서 실행 중입니다.")

Mounted at /content/drive


In [3]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 드라이브 마운트 작업 후 아래 경로에서 폰트 가져오기
# /content/drive/MyDrive/fonts/NanumGothic.ttf

font_path = f'/content/drive/MyDrive/fonts/NanumGothic.ttf'  # 설치한 폰트 경로
fm.fontManager.addfont(font_path)  # 폰트 경로 추가

plt.rcParams['font.family'] = 'NanumGothic'  # 사용 폰트 입력
plt.rcParams['axes.unicode_minus'] = False   # 음수 부호 사용

In [4]:
df = pd.read_csv("/content/drive/MyDrive/dataset/mission15_train.csv")
print("shape:", df.shape)
df.head()

shape: (7000, 6)


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,6,73,No,7,2,58.0
1,1,89,Yes,7,2,64.0
2,3,97,Yes,8,0,75.0
3,8,70,No,5,5,59.0
4,7,94,Yes,7,4,86.0


## 2. 기초 탐색 (EDA)
데이터 타입, 결측치, 기술통계를 확인한다.

In [5]:
df.info()
print("\n결측치:\n", df.isnull().sum())
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     7000 non-null   int64  
 1   Previous Scores                   7000 non-null   int64  
 2   Extracurricular Activities        7000 non-null   object 
 3   Sleep Hours                       7000 non-null   int64  
 4   Sample Question Papers Practiced  7000 non-null   int64  
 5   Performance Index                 7000 non-null   float64
dtypes: float64(1), int64(4), object(1)
memory usage: 328.3+ KB

결측치:
 Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64


,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,7000.000000,7000.000000,7000.000000,7000.000000,7000.000000
mean,4.950000,69.429714,6.530571,4.607429,55.095143
std,2.590621,17.289197,1.696144,2.863550,19.151574
min,1.000000,40.000000,4.000000,0.000000,10.000000
25%,3.000000,54.000000,5.000000,2.000000,40.000000
50%,5.000000,69.000000,7.000000,5.000000,55.000000
75%,7.000000,85.000000,8.000000,7.000000,70.000000
max,9.000000,99.000000,9.000000,9.000000,100.000000


### 2-1. 범주형 변수 분포
`Extracurricular Activities` 는 Yes/No 두 값을 가지며 거의 균등하게 분포한다.

In [6]:
df["Extracurricular Activities"].value_counts()

,count
Extracurricular Activities,
No,3522
Yes,3478


### 2-2. 목표변수와의 상관관계
범주형을 0/1 로 임시 변환하여 상관계수를 확인한다.  
`Previous Scores`(≈0.91) 가 `Performance Index` 와 압도적으로 강한 양의 상관을 보이고,
`Hours Studied`(≈0.37) 가 그 다음이다. 나머지 세 변수의 선형 상관은 약하다.

In [7]:
tmp = df.copy()
tmp["Extracurricular Activities"] = tmp["Extracurricular Activities"].map({"Yes": 1, "No": 0})
corr = tmp.corr()["Performance Index"].sort_values(ascending=False)
corr

,Performance Index
Performance Index,1.000000
Previous Scores,0.913866
Hours Studied,0.374150
Sample Question Papers Practiced,0.049908
Sleep Hours,0.049278
Extracurricular Activities,0.025306


## 3. 전처리 설계
- **결측치:** 없음 → 별도 처리 불필요
- **범주형 `Extracurricular Activities`:** `OrdinalEncoder` 로 No→0, Yes→1 인코딩
- **수치형 5개:** 선형회귀는 스케일에 불변이므로 그대로 통과(passthrough)

전처리와 모델을 하나의 `Pipeline` 으로 묶어 학습·저장·추론 시 동일한 변환이 적용되도록 한다.  
→ 연구자 2 는 `pipe.predict(원본 test.csv)` 만으로 추론이 가능해진다.

In [8]:
TARGET = "Performance Index"
CATEGORICAL = ["Extracurricular Activities"]

X = df.drop(columns=[TARGET])
y = df[TARGET]

pre = ColumnTransformer(
    transformers=[("cat", OrdinalEncoder(categories=[["No", "Yes"]]), CATEGORICAL)],
    remainder="passthrough",
)
pipe = Pipeline([("pre", pre), ("reg", LinearRegression())])
pipe

Pipeline(steps=[('pre',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OrdinalEncoder(categories=[['No',
                                                                              'Yes']]),
                                                  ['Extracurricular '
                                                   'Activities'])])),
                ('reg', LinearRegression())])

## 4. 모델링 및 RMSE 평가
학습:검증 = 8:2 로 나누어 검증 RMSE 를 측정한다.

참고: https://wikidocs.net/291548

In [9]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_tr, y_tr)

val_pred = pipe.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_pred))
r2 = r2_score(y_val, val_pred)
print(f"검증 RMSE = {rmse:.4f}")
print(f"검증 R^2  = {r2:.4f}")

검증 RMSE = 2.0103
검증 R^2  = 0.9893


### 4-1. 회귀 계수 해석
`Previous Scores` 와 `Hours Studied` 의 계수가 크게 나와 EDA 의 상관분석 결과와 일치한다.

In [10]:
feat_names = CATEGORICAL + [c for c in X.columns if c not in CATEGORICAL]
coef = pipe.named_steps["reg"].coef_
for name, c in zip(feat_names, coef):
    print(f"{name:35s} {c:8.4f}")
print(f"{'intercept':35s} {pipe.named_steps['reg'].intercept_:8.4f}")

Extracurricular Activities            0.6234
Hours Studied                         2.8627
Previous Scores                       1.0163
Sleep Hours                           0.4827
Sample Question Papers Practiced      0.1913
intercept                           -33.9778


## 5. 전체 데이터 재학습 후 model.pkl 저장
검증이 끝났으므로 전체 train 데이터로 다시 학습하여 최종 모델을 저장한다.

In [11]:
pipe.fit(X, y)
joblib.dump(pipe, "model.pkl")
print("model.pkl 저장 완료")

model.pkl 저장 완료


## 6. 정리
- 결측치 없음, 범주형 1개(OrdinalEncoder)·수치형 5개(passthrough)
- **선형회귀 검증 RMSE ≈ 2.01, R² ≈ 0.989** — 목표변수 범위(10~100) 대비 매우 우수
- 핵심 변수: `Previous Scores` > `Hours Studied`
- 이 파이프라인을 `train.py` 로 옮겨 도커 이미지로 자동화한다.